# National Dashboard + Part Lookup Sheet Generator

Builds two sheets on top of an FD Processed workbook (same template as
`FD_Processed_Cummins_Steron_BIN_*.xlsx`):

1. **National Dashboard** (first tab) — a filterable national roll-up.
   - Three numeric filter fields (Min/Max): **Total Calls**, **MM06**, **MM12**.
     These act like a slicer — adjust the Min/Max cells and every total below recalculates.
   - Monthly subtotal table: `Amt. Est. OO` and `Amt. Sched. Order`, subtotaled **per month**
     (M-1 ... M-9) across every part number that falls inside the current filter ranges.
   - A "Parts Included" count and combined totals so you can see the filtered scope at a glance.

2. **Part Lookup** (second tab) — the single-part dropdown view built previously, now with a
   second dropdown, **Select Branch**, next to the P/N selector: choose **National** to sum
   Demand & Calls across every branch in the pmovdcE sheet, or pick a specific Brc to see that
   branch alone. Both feed the **Demand & Calls Trend** line chart (D-1..D-16 / C-1..C-12)
   sourced from the pmovdcE sheet below.

3. **pmovdcE** (third tab) — a plain copy of `Brc`, `P/N`, `D-1..D-16`, `C-1..C-12` from the
   pmovdcE parts-movement export you provide, plus a small branch list off to the side that
   backs the Part Lookup branch dropdown. If that export has multiple tabs (e.g. a Steron
   subset, a Non-Steron subset, and a combined tab), this notebook auto-picks whichever tab has
   the most data rows, since the combined tab is a superset of any subset tabs.

> **Why Min/Max fields instead of a native Excel Slicer:** true Excel Slicers only attach to
> PivotTables/Tables and are built for categorical fields (they show one button per distinct
> value). `Total Calls`, `MM06`, and `MM12` are continuous numeric fields with hundreds of
> distinct values each, so a real slicer would be an unusable wall of buttons. A Min/Max range
> pair gives you the same "filter the view" behavior and is the standard way to slice a
> continuous field in Excel. If you'd rather have a real PivotTable+Slicer for a different,
> categorical field (e.g. RC, Alert), that's a quick add — just say the word.

Both sheets are driven by `INDEX`/`MATCH` / `SUMIFS` formulas against column **headers** and
resolved column letters — not fixed positions — so this keeps working as long as the source
file keeps the same header names, regardless of row count or minor column reordering.

In [1]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.chart import LineChart, Reference
from openpyxl.utils import get_column_letter
from openpyxl.chart.label import DataLabelList
from openpyxl.chart.marker import Marker
from openpyxl.utils import get_column_letter, column_index_from_string

In [2]:
# ============================================================
# CONFIG — change these per run
# ============================================================
INPUT_FILE = "FD Processed Non Steron CNH 19Aug26.xlsx"   # input workbook (same template each time)
OUTPUT_FILE = "FD Processed Non Steron CNH Dashboard 19Aug26.xlsx"               # where to save the result

SOURCE_SHEET_NAME = "FD Processed"        # tab holding the raw FD Processed data
NATIONAL_SHEET_NAME = "National Dashboard"  # tab this script creates/replaces (1st tab)
LOOKUP_SHEET_NAME = "Part Lookup"           # tab this script creates/replaces (2nd tab)

PN_COLUMN_HEADER = "P/N"             # header of the part-number column in the source sheet
DESC_COLUMN_HEADER = "Desc"

# Fields shown in the Part Lookup "Part Summary" strip — (display label, exact source header text)
SUMMARY_FIELDS = [
    ("RC", "RC"),
    ("Total Calls", "Total Calls"),
    ("Unit Price", "DN Price"),
    ("FD Final", "FD_final"),
    ("Max", "Max"),
    ("On Hand", "OH"),
    ("On Order", "OO"),
    ("Trend Coef", "Trend Coef"),
    ("%Accum.", "%Accum."),
    ("Alert Score", "forecast_alert_score"),
    ("Alert", "forecast_alert_label"),
]

# Monthly metrics grid — row label must be the exact prefix used in the source headers
# (e.g. source column "Est. OH M-3" = "Est. OH" + " " + "M-3")
UNIT_METRICS = ["Incoming", "Est. OH", "Est. OO", "Sched. Order"]
AMOUNT_METRICS = ["Amt. Est. OO", "Amt. Sched. Order"]
MONTH_LABELS = [f"M-{i}" for i in range(1, 10)]   # M-1 .. M-9 — covers the widest metric (Est. OH)

# Fields the National Dashboard filters (slices) on — exact source header text
FILTER_FIELDS = ["Total Calls", "MM06", "MM12"]

# ---- pmovdcE movement data (demand & calls history) ----
PMOVDCE_FILE = "pmovdcE CNH Non Steron 19Aug26.xlsx"   # separate input file — parts movement report
MOVEMENT_SHEET_NAME = "pmovdcE"                        # tab this script creates in the output workbook
BRC_HEADER = "Brc"
DEMAND_PREFIX = "D-"     # source columns: D-1 .. D-16 (D-1 = most recent month)
CALL_PREFIX = "C-"       # source columns: C-1 .. C-12 (C-1 = most recent month)
N_DEMAND_MONTHS = 16
N_CALL_MONTHS = 12

# ---- DN Price comparison file (source of the "Source"/Agc for Part Summary) ----
USE_PRICE_COMPARISON_FILE = True   # set False to skip it entirely -- no file needed, no lookup
                                    # sheet built, and the Part Summary "Source" cell is left as
                                    # a static "N/A" instead of a formula. Every other column
                                    # (RC, Total Calls, Unit Price, FD Final, etc.) is untouched
                                    # either way -- those all come from the FD Processed sheet.
PRICE_COMPARISON_FILE = "Comparison DN Price CNH 18Aug26.xlsx"
PRICE_COMPARISON_SHEET = "Price List"
PRICE_COMPARISON_HEADER_ROW = 2     # 1-indexed row with the 'PN' / 'Agc' / 'DN Price' headers
PRICE_COMPARISON_PN_COL = "V"
PRICE_COMPARISON_AGC_COL = "W"
AGC_LOOKUP_SHEET_NAME = "DN Price Agc"   # hidden helper sheet this script builds


In [3]:
wb = openpyxl.load_workbook(INPUT_FILE, data_only=False)
src = wb[SOURCE_SHEET_NAME]

last_row = src.max_row               # last row of data (header + parts)
last_col_letter = get_column_letter(src.max_column)   # last column, e.g. "BO"
default_pn = str(src.cell(row=2, column=1).value)     # first P/N in the sheet, used as the default dropdown value

print(f"Source sheet: {SOURCE_SHEET_NAME!r}  |  rows: {last_row}  |  columns: A:{last_col_letter}  |  default P/N: {default_pn}")


def find_column_letter(ws, header_text, required=True):
    """Return the column letter whose row-1 header exactly matches `header_text`."""
    for cell in ws[1]:
        if cell.value == header_text:
            return get_column_letter(cell.column)
    if required:
        raise ValueError(f'Required column "{header_text}" not found in {ws.title!r} header row.')
    return None


def column_values(ws, col_letter, last_row):
    """Numeric values in a column, rows 2..last_row (skips blanks/non-numeric)."""
    vals = []
    for r in range(2, last_row + 1):
        v = ws[f"{col_letter}{r}"].value
        if isinstance(v, (int, float)):
            vals.append(v)
    return vals


def find_movement_sheet(pmov_wb, pn_header=PN_COLUMN_HEADER, max_header_scan_rows=15):
    """
    A pmovdcE export can contain several tabs (e.g. a Steron subset, a Non-Steron subset,
    and a combined tab). We want the combined one, so: scan every sheet's first few rows for
    a header row containing `pn_header`, and pick whichever sheet has the most data rows
    below its header — the combined tab is a superset of any subset tabs.
    Returns (worksheet, header_row_index).
    """
    best = None  # (worksheet, header_row_index, data_row_count)
    for sheet in pmov_wb.worksheets:
        for r in range(1, min(max_header_scan_rows, sheet.max_row) + 1):
            row_vals = [sheet.cell(row=r, column=c).value for c in range(1, sheet.max_column + 1)]
            if pn_header in row_vals:
                data_rows = sheet.max_row - r
                if best is None or data_rows > best[2]:
                    best = (sheet, r, data_rows)
                break
    if best is None:
        raise ValueError(f'No sheet with a "{pn_header}" header found in {pmov_wb.sheetnames}')
    return best[0], best[1]


def find_column_index(ws, header_row, header_text):
    """Like find_column_letter, but for a header that may not be on row 1."""
    for cell in ws[header_row]:
        if cell.value == header_text:
            return cell.column
    return None


# Drop any previous version of the sheets we build, so re-running this notebook is safe
for sheet_name in (NATIONAL_SHEET_NAME, LOOKUP_SHEET_NAME):
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]


Source sheet: 'FD Processed'  |  rows: 13289  |  columns: A:BO  |  default P/N: 51508762


In [4]:
# ============================================================
# Styles
# ============================================================
NAVY = "1F2A3A"
LIGHT_GREY = "F4F6F8"
MONTH_BLUE = "D9E2EC"

white_font = Font(color="FFFFFF", bold=True, size=13, name="Arial")
title_fill = PatternFill("solid", fgColor=NAVY)

label_font = Font(bold=True, name="Arial", size=10, color="33414F")
value_font = Font(name="Arial", size=10, color="000000")

section_font = Font(bold=True, name="Arial", size=10, color="FFFFFF")
section_fill = PatternFill("solid", fgColor="33414F")

lightgrey_fill = PatternFill("solid", fgColor=LIGHT_GREY)
month_fill = PatternFill("solid", fgColor=MONTH_BLUE)

thin = Side(style="thin", color="C9D2DA")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

input_fill = PatternFill("solid", fgColor="FFF6E5")
input_font = Font(bold=True, name="Arial", size=12, color="000000")

total_font = Font(bold=True, name="Arial", size=10, color="000000")
total_fill = PatternFill("solid", fgColor="EDEFF2")


def set_cell(ws, cell_ref, value, font=None, fill=None, align=None, numfmt=None, brd=None):
    """Write a value/formula into a cell with optional styling."""
    c = ws[cell_ref]
    c.value = value
    if font: c.font = font
    if fill: c.fill = fill
    if align: c.alignment = align
    if numfmt: c.number_format = numfmt
    if brd: c.border = brd
    return c


def two_way_lookup(row_key_ref, header_text_expr, source_sheet=SOURCE_SHEET_NAME,
                    last_col_letter=None, row_match_col="A"):
    """
    Build an =IFERROR(INDEX(...), "") formula that looks up `row_key_ref` (a P/N, e.g. "$B$4")
    down `source_sheet`'s `row_match_col` column, and `header_text_expr` (a formula fragment
    that evaluates to an exact source header string, e.g. '"RC"' or '$A12&" "&B$11')
    across the source sheet's header row (row 1).
    """
    return (
        f'=IFERROR(INDEX(\'{source_sheet}\'!$A:${last_col_letter}, '
        f'MATCH({row_key_ref},\'{source_sheet}\'!${row_match_col}:${row_match_col},0), '
        f'MATCH({header_text_expr},\'{source_sheet}\'!$A$1:${last_col_letter}$1,0)), "")'
    )


def sumifs_formula(sum_col_letter, criteria_cols_and_bounds, source_sheet, last_row):
    """
    Build a SUMIFS formula summing `sum_col_letter` over rows 2..last_row where each
    (col_letter, min_ref, max_ref) in `criteria_cols_and_bounds` is satisfied.
    """
    parts = [f"'{source_sheet}'!${sum_col_letter}$2:${sum_col_letter}${last_row}"]
    for col_letter, min_ref, max_ref in criteria_cols_and_bounds:
        rng = f"'{source_sheet}'!${col_letter}$2:${col_letter}${last_row}"
        parts.append(f'{rng}, ">="&{min_ref}')
        parts.append(f'{rng}, "<="&{max_ref}')
    return "=SUMIFS(" + ", ".join(parts) + ")"


def countifs_formula(count_col_letter, criteria_cols_and_bounds, source_sheet, last_row):
    """Same criteria pattern as sumifs_formula, but a COUNTIFS (used for 'Parts Included')."""
    parts = []
    for col_letter, min_ref, max_ref in criteria_cols_and_bounds:
        rng = f"'{source_sheet}'!${col_letter}$2:${col_letter}${last_row}"
        parts.append(f'{rng}, ">="&{min_ref}')
        parts.append(f'{rng}, "<="&{max_ref}')
    return "=COUNTIFS(" + ", ".join(parts) + ")"


In [5]:
# ============================================================
# NATIONAL DASHBOARD  (built first -> ends up as the first tab)
# ============================================================
nat = wb.create_sheet(NATIONAL_SHEET_NAME, 0)

# ---- resolve source columns we need ----
calls_col = find_column_letter(src, "Total Calls")
mm06_col = find_column_letter(src, "MM06")
mm12_col = find_column_letter(src, "MM12")

# Amt. Est. OO / Amt. Sched. Order — only the months that actually exist in this template
amt_month_cols = {}   # {metric: {month_label: col_letter}}
for metric in AMOUNT_METRICS:
    amt_month_cols[metric] = {}
    for m in MONTH_LABELS:
        col = find_column_letter(src, f"{metric} {m}", required=False)
        if col:
            amt_month_cols[metric][m] = col

# ---- title ----
nat.merge_cells("A1:K2")
set_cell(nat, "A1", "NATIONAL DASHBOARD", white_font, title_fill,
          Alignment(vertical="center", horizontal="left", indent=1))
for row in nat["A1:K2"]:
    for c in row:
        c.fill = title_fill

# ---- filters (numeric slicer) ----
nat.merge_cells("A4:F4")
set_cell(nat, "A4", "FILTERS  (acts as a numeric slicer — adjust Min / Max)", section_font, section_fill)

set_cell(nat, "A5", "Field", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
set_cell(nat, "B5", "Min", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
set_cell(nat, "C5", "Max", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)

filter_col_letters = {"Total Calls": calls_col, "MM06": mm06_col, "MM12": mm12_col}
filter_cell_refs = {}   # field -> (min_ref, max_ref)
for i, field in enumerate(FILTER_FIELDS):
    r = 6 + i
    col_letter = filter_col_letters[field]
    vals = column_values(src, col_letter, last_row)
    vmin, vmax = (min(vals), max(vals)) if vals else (0, 0)
    set_cell(nat, f"A{r}", field, label_font, lightgrey_fill, Alignment(horizontal="left", indent=1), brd=border)
    set_cell(nat, f"B{r}", vmin, input_font, input_fill, Alignment(horizontal="center"), numfmt="#,##0", brd=border)
    set_cell(nat, f"C{r}", vmax, input_font, input_fill, Alignment(horizontal="center"), numfmt="#,##0", brd=border)
    filter_cell_refs[field] = (f"$B${r}", f"$C${r}")

criteria = [
    (calls_col, *filter_cell_refs["Total Calls"]),
    (mm06_col, *filter_cell_refs["MM06"]),
    (mm12_col, *filter_cell_refs["MM12"]),
]

# ---- summary strip (filtered) ----
summary_row_label = 6 + len(FILTER_FIELDS) + 1     # blank row, then section header
sec_row = summary_row_label
nat.merge_cells(f"A{sec_row}:F{sec_row}")
set_cell(nat, f"A{sec_row}", "SUMMARY", section_font, section_fill)

lbl_row = sec_row + 1
val_row = sec_row + 2
pn_col = find_column_letter(src, PN_COLUMN_HEADER)

set_cell(nat, f"A{lbl_row}", "PN Count", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
set_cell(nat, f"B{lbl_row}", "Total Amt. Est. OO", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
set_cell(nat, f"C{lbl_row}", "Total Amt. Sched. Order", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
set_cell(nat, f"D{lbl_row}", "Combined Total", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)

parts_included_formula = countifs_formula(pn_col, criteria, SOURCE_SHEET_NAME, last_row)
set_cell(nat, f"A{val_row}", parts_included_formula, value_font, None, Alignment(horizontal="center"), numfmt="#,##0", brd=border)
# B/C/D values are filled in after the monthly table below (they reference its Total column)


<Cell 'National Dashboard'.A12>

In [6]:
# ---- monthly subtotal table (filtered) ----
table_sec_row = val_row + 2
nat.merge_cells(f"A{table_sec_row}:K{table_sec_row}")
set_cell(nat, f"A{table_sec_row}", "MONTHLY SUBTOTALS", section_font, section_fill)

header_row = table_sec_row + 1
set_cell(nat, f"A{header_row}", "Metric", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
for i, m in enumerate(MONTH_LABELS):
    col = get_column_letter(2 + i)   # B..J
    set_cell(nat, f"{col}{header_row}", m, label_font, month_fill, Alignment(horizontal="center"), brd=border)
set_cell(nat, f"K{header_row}", "Total", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)

metric_rows = {}
row = header_row + 1
for metric in AMOUNT_METRICS:
    metric_rows[metric] = row
    set_cell(nat, f"A{row}", metric, label_font, lightgrey_fill, Alignment(horizontal="left", indent=1), brd=border)
    month_cells = []
    for i, m in enumerate(MONTH_LABELS):
        col = get_column_letter(2 + i)
        amt_col = amt_month_cols[metric].get(m)
        if amt_col:
            formula = sumifs_formula(amt_col, criteria, SOURCE_SHEET_NAME, last_row)
            set_cell(nat, f"{col}{row}", formula, value_font, None, Alignment(horizontal="center"),
                      numfmt="$#,##0", brd=border)
            month_cells.append(f"{col}{row}")
        else:
            set_cell(nat, f"{col}{row}", "", value_font, None, Alignment(horizontal="center"), brd=border)
    total_formula = "=" + "+".join(month_cells) if month_cells else 0
    set_cell(nat, f"K{row}", total_formula, total_font, total_fill, Alignment(horizontal="center"),
              numfmt="$#,##0", brd=border)
    row += 1

# combined total row (Amt. Est. OO + Amt. Sched. Order, per month + grand total)
combined_row = row
set_cell(nat, f"A{combined_row}", "Combined Total", total_font, total_fill, Alignment(horizontal="left", indent=1), brd=border)
metric_row_refs = list(metric_rows.values())
for i, m in enumerate(MONTH_LABELS):
    col = get_column_letter(2 + i)
    formula = "=" + "+".join(f"{col}{r}" for r in metric_row_refs)
    set_cell(nat, f"{col}{combined_row}", formula, total_font, total_fill, Alignment(horizontal="center"),
              numfmt="$#,##0", brd=border)
grand_total_formula = "=" + "+".join(f"K{r}" for r in metric_row_refs)
set_cell(nat, f"K{combined_row}", grand_total_formula, total_font, total_fill, Alignment(horizontal="center"),
          numfmt="$#,##0", brd=border)

# ---- back-fill the summary strip totals now that row numbers are known ----
est_oo_row = metric_rows["Amt. Est. OO"]
sched_row = metric_rows["Amt. Sched. Order"]
set_cell(nat, f"B{val_row}", f"=K{est_oo_row}", value_font, None, Alignment(horizontal="center"), numfmt="$#,##0", brd=border)
set_cell(nat, f"C{val_row}", f"=K{sched_row}", value_font, None, Alignment(horizontal="center"), numfmt="$#,##0", brd=border)
set_cell(nat, f"D{val_row}", f"=K{combined_row}", value_font, None, Alignment(horizontal="center"), numfmt="$#,##0", brd=border)

# ---- column widths ----
nat.column_dimensions["A"].width = 22
for col in "BCDEFGHIJK":
    nat.column_dimensions[col].width = 13

national_last_row = combined_row
print(f"National Dashboard built through row {national_last_row}.")


National Dashboard built through row 18.


In [7]:
# ============================================================
# PART LOOKUP  (built second -> ends up as the second tab)
# ============================================================
ws = wb.create_sheet(LOOKUP_SHEET_NAME, 1)

ws.merge_cells("A1:K2")
set_cell(ws, "A1", "PART NUMBER LOOKUP", white_font, title_fill,
          Alignment(vertical="center", horizontal="left", indent=1))
for row in ws["A1:K2"]:
    for c in row:
        c.fill = title_fill

set_cell(ws, "A4", "Select P/N:", label_font)
set_cell(ws, "B4", default_pn, input_font, input_fill, Alignment(horizontal="center"), brd=border)
ws["B4"].number_format = "@"   # keep as text — the source P/N column is text, not numeric

set_cell(ws, "C4", "Select Branch:", label_font)
set_cell(ws, "D4", "National", input_font, input_fill, Alignment(horizontal="center"), brd=border)
ws["D4"].number_format = "@"
# Data validation for D4 (National + each Brc) is added later, once the pmovdcE sheet
# (and its branch list) has been built.

set_cell(ws, "E4", "Description:", label_font)
ws.merge_cells("F4:K4")
desc_formula = two_way_lookup("$B$4", f'"{DESC_COLUMN_HEADER}"', last_col_letter=last_col_letter)
desc_formula = desc_formula.replace('"")', '"Part not found")')  # friendlier fallback text
set_cell(ws, "F4", desc_formula, Font(italic=True, name="Arial", size=10))

# Dropdown of every P/N in the source sheet
dv = DataValidation(type="list", formula1=f"='{SOURCE_SHEET_NAME}'!$A$2:$A${last_row}",
                     allow_blank=False, showDropDown=False)
dv.error = f"Please select a valid P/N from the {SOURCE_SHEET_NAME} sheet."
dv.errorTitle = "Invalid P/N"
ws.add_data_validation(dv)
dv.add(ws["B4"])


In [8]:
# ============================================================
# Agc lookup — copy P/N + Agc from the DN Price comparison file, filtered to this run's P/Ns
# ============================================================
if USE_PRICE_COMPARISON_FILE:
    price_wb = openpyxl.load_workbook(PRICE_COMPARISON_FILE, data_only=True, read_only=True)
    price_src = price_wb[PRICE_COMPARISON_SHEET]

    pn_col_idx = column_index_from_string(PRICE_COMPARISON_PN_COL)
    agc_col_idx = column_index_from_string(PRICE_COMPARISON_AGC_COL)

    # Only keep rows for P/Ns actually in this run's source sheet — keeps the helper sheet small
    source_pns = {str(src.cell(row=r, column=1).value).strip().upper() for r in range(2, last_row + 1)}

    agc_rows = []
    for row in price_src.iter_rows(min_row=PRICE_COMPARISON_HEADER_ROW + 1, values_only=True):
        pn = row[pn_col_idx - 1]
        if pn is None:
            continue
        pn_key = str(pn).strip().upper()
        if pn_key in source_pns:
            agc_rows.append((pn_key, row[agc_col_idx - 1]))

    price_wb.close()

    if AGC_LOOKUP_SHEET_NAME in wb.sheetnames:
        del wb[AGC_LOOKUP_SHEET_NAME]
    agc_ws = wb.create_sheet(AGC_LOOKUP_SHEET_NAME)
    agc_ws.sheet_state = "hidden"

    set_cell(agc_ws, "A1", "P/N", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
    set_cell(agc_ws, "B1", "Agc", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
    for i, (pn, agc) in enumerate(agc_rows, start=2):
        agc_ws.cell(row=i, column=1, value=pn)
        agc_ws.cell(row=i, column=2, value=agc)

    agc_lookup_last_row = 1 + len(agc_rows)
    print(f"Agc lookup: {len(agc_rows)} of {len(source_pns)} source P/Ns matched in {PRICE_COMPARISON_FILE}")
else:
    agc_lookup_last_row = None
    print("USE_PRICE_COMPARISON_FILE = False -- skipping price comparison file, no lookup sheet built.")


Agc lookup: 12915 of 13288 source P/Ns matched in Comparison DN Price CNH 18Aug26.xlsx


In [9]:
# ============================================================
# Part Summary section
# ============================================================
ws.merge_cells("A6:K6")
set_cell(ws, "A6", "PART SUMMARY", section_font, section_fill)

# ---- Source (Agc) — from the DN Price comparison file, not the FD Processed sheet ----
set_cell(ws, "A7", "Source", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
if USE_PRICE_COMPARISON_FILE:
    agc_formula = (
        f"=IFERROR(INDEX('{AGC_LOOKUP_SHEET_NAME}'!$B$2:$B${agc_lookup_last_row}, "
        f"MATCH($B$4, '{AGC_LOOKUP_SHEET_NAME}'!$A$2:$A${agc_lookup_last_row}, 0)), \"Review\")"
    )
    set_cell(ws, "A8", agc_formula, value_font, None, Alignment(horizontal="center"), brd=border)
else:
    set_cell(ws, "A8", "N/A", value_font, None, Alignment(horizontal="center"), brd=border)

for i, (label, header) in enumerate(SUMMARY_FIELDS):
    col = get_column_letter(2 + i)   # shifted +1 to make room for Source in column A
    set_cell(ws, f"{col}7", label, label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
    formula = two_way_lookup("$B$4", f'"{header}"', last_col_letter=last_col_letter)
    numfmt = None
    if header == "DN Price": numfmt = "$#,##0.00"
    elif header == "%Accum.": numfmt = '0.0"%"'
    elif header == "Trend Coef": numfmt = "0.00"
    set_cell(ws, f"{col}8", formula, value_font, None, Alignment(horizontal="center"), numfmt=numfmt, brd=border)

In [10]:
# ============================================================
# Monthly Detail — Units
# ============================================================
ws.merge_cells("A10:J10")
set_cell(ws, "A10", "MONTHLY DETAIL QTY", section_font, section_fill)

set_cell(ws, "A11", "Metric", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
for i, m in enumerate(MONTH_LABELS):
    col = get_column_letter(2 + i)
    set_cell(ws, f"{col}11", m, label_font, month_fill, Alignment(horizontal="center"), brd=border)

row = 12
for metric in UNIT_METRICS:
    set_cell(ws, f"A{row}", metric, label_font, lightgrey_fill, Alignment(horizontal="left", indent=1), brd=border)
    for i, m in enumerate(MONTH_LABELS):
        col = get_column_letter(2 + i)
        header_expr = f'$A{row}&" "&{col}$11'   # e.g. "Est. OH"&" "&"M-3" -> "Est. OH M-3"
        formula = two_way_lookup("$B$4", header_expr, last_col_letter=last_col_letter)
        set_cell(ws, f"{col}{row}", formula, value_font, None, Alignment(horizontal="center"),
                  numfmt="#,##0", brd=border)
    row += 1

units_last_row = row - 1   # last metric row, needed later for the chart


In [11]:
# ============================================================
# Monthly Detail — Dollar Amounts
# ============================================================
amt_header_row = units_last_row + 2
ws.merge_cells(f"A{amt_header_row}:J{amt_header_row}")
set_cell(ws, f"A{amt_header_row}", "MONTHLY DETAIL", section_font, section_fill)

col_header_row = amt_header_row + 1
set_cell(ws, f"A{col_header_row}", "Metric", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
for i, m in enumerate(MONTH_LABELS):
    col = get_column_letter(2 + i)
    set_cell(ws, f"{col}{col_header_row}", m, label_font, month_fill, Alignment(horizontal="center"), brd=border)

row = col_header_row + 1
for metric in AMOUNT_METRICS:
    set_cell(ws, f"A{row}", metric, label_font, lightgrey_fill, Alignment(horizontal="left", indent=1), brd=border)
    for i, m in enumerate(MONTH_LABELS):
        col = get_column_letter(2 + i)
        header_expr = f'$A{row}&" "&{col}${col_header_row}'
        formula = two_way_lookup("$B$4", header_expr, last_col_letter=last_col_letter)
        set_cell(ws, f"{col}{row}", formula, value_font, None, Alignment(horizontal="center"),
                  numfmt="$#,##0.00", brd=border)
    row += 1

amounts_last_row = row - 1


In [12]:
# ============================================================
# Trend chart — units across months for the selected part
# ============================================================
chart_anchor_row = amounts_last_row + 2

chart = LineChart()
chart.title = "Incoming / Est. OH / Est. OO / Sched. Order"
chart.style = 2
chart.y_axis.title = "Qty"
chart.x_axis.title = "Month"
chart.width = 24
chart.height = 10

cats = Reference(ws, min_col=2, max_col=1 + len(MONTH_LABELS), min_row=11, max_row=11)
data = Reference(ws, min_col=1, max_col=1 + len(MONTH_LABELS), min_row=12, max_row=units_last_row)
chart.add_data(data, titles_from_data=True, from_rows=True)
chart.set_categories(cats)

ws.add_chart(chart, f"A{chart_anchor_row}")

# Column widths
ws.column_dimensions["A"].width = 20
for col in "BCDEFGHIJK":
    ws.column_dimensions[col].width = 13


In [13]:
# ============================================================
# pmovdcE — copy Brc / P/N / D-i / C-i into their own sheet
# ============================================================
pmov_wb = openpyxl.load_workbook(PMOVDCE_FILE, data_only=True)
mov_src, mov_header_row = find_movement_sheet(pmov_wb)
print(f'Using sheet {mov_src.title!r} in {PMOVDCE_FILE!r}  (header row {mov_header_row}, '
      f'{mov_src.max_row - mov_header_row} data rows)')

brc_idx = find_column_index(mov_src, mov_header_row, BRC_HEADER)
pn_idx = find_column_index(mov_src, mov_header_row, PN_COLUMN_HEADER)
demand_idxs = [find_column_index(mov_src, mov_header_row, f"{DEMAND_PREFIX}{i}") for i in range(1, N_DEMAND_MONTHS + 1)]
call_idxs = [find_column_index(mov_src, mov_header_row, f"{CALL_PREFIX}{i}") for i in range(1, N_CALL_MONTHS + 1)]

missing = []
if brc_idx is None: missing.append(BRC_HEADER)
if pn_idx is None: missing.append(PN_COLUMN_HEADER)
missing += [f"{DEMAND_PREFIX}{i}" for i, idx in zip(range(1, N_DEMAND_MONTHS + 1), demand_idxs) if idx is None]
missing += [f"{CALL_PREFIX}{i}" for i, idx in zip(range(1, N_CALL_MONTHS + 1), call_idxs) if idx is None]
if missing:
    raise ValueError(f"pmovdcE sheet is missing expected column(s): {missing}")

if MOVEMENT_SHEET_NAME in wb.sheetnames:
    del wb[MOVEMENT_SHEET_NAME]
mov = wb.create_sheet(MOVEMENT_SHEET_NAME, 2)   # 3rd tab: National Dashboard, Part Lookup, pmovdcE

# header row
mov_headers = [BRC_HEADER, PN_COLUMN_HEADER] + [f"{DEMAND_PREFIX}{i}" for i in range(1, N_DEMAND_MONTHS + 1)] \
              + [f"{CALL_PREFIX}{i}" for i in range(1, N_CALL_MONTHS + 1)]
for c, h in enumerate(mov_headers, start=1):
    set_cell(mov, f"{get_column_letter(c)}1", h, label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)

# data rows — copied as plain values (source is a raw movement export, not a formula sheet)
all_idxs = [brc_idx, pn_idx] + demand_idxs + call_idxs
out_row = 2
brc_values_seen = set()
for r in range(mov_header_row + 1, mov_src.max_row + 1):
    row_vals = [mov_src.cell(row=r, column=idx).value for idx in all_idxs]
    if row_vals[1] in (None, ""):   # skip any blank P/N row
        continue
    for c, v in enumerate(row_vals, start=1):
        mov.cell(row=out_row, column=c, value=v)
    if row_vals[0] not in (None, ""):
        brc_values_seen.add(str(row_vals[0]))
    out_row += 1

mov_last_row = out_row - 1
mov_last_col_letter = get_column_letter(len(mov_headers))
mov.freeze_panes = "A2"
mov.column_dimensions["A"].width = 8
mov.column_dimensions["B"].width = 16

# ---- branch filter list: "National" + every distinct Brc, written a couple columns past
#      the data so it can back a dropdown on Part Lookup ($D$4) ----
branch_list = ["National"] + sorted(brc_values_seen)
branch_list_col = len(mov_headers) + 2   # one blank spacer column after the data
branch_list_col_letter = get_column_letter(branch_list_col)
set_cell(mov, f"{branch_list_col_letter}1", "Branch Filter List", label_font, lightgrey_fill,
          Alignment(horizontal="center"), brd=border)
for i, b in enumerate(branch_list):
    mov.cell(row=2 + i, column=branch_list_col, value=b)
branch_list_last_row = 1 + len(branch_list)
mov.column_dimensions[branch_list_col_letter].width = 16

# ---- attach the dropdown + default value on Part Lookup now that the list exists ----
ws["D4"].value = "National"
branch_dv = DataValidation(
    type="list",
    formula1=f"='{MOVEMENT_SHEET_NAME}'!${branch_list_col_letter}$2:${branch_list_col_letter}${branch_list_last_row}",
    allow_blank=False, showDropDown=False)
branch_dv.error = 'Please select "National" or a valid Brc from the pmovdcE sheet.'
branch_dv.errorTitle = "Invalid Branch"
ws.add_data_validation(branch_dv)
branch_dv.add(ws["D4"])

print(f"pmovdcE sheet: {mov_last_row - 1} rows copied  |  columns A:{mov_last_col_letter}  |  branches found: {sorted(brc_values_seen)}")


Using sheet 'Non Steron' in 'pmovdcE CNH Non Steron 19Aug26.xlsx'  (header row 5, 40318 data rows)
pmovdcE sheet: 40318 rows copied  |  columns A:AD  |  branches found: ['20', '21', '22', '23', '24', '25', '26', '27', '28', '30', '31', '32', '37', '38', '39', '40', '41', '43', '45', '47', '50', '53', '55', '57', '61', '62', '63', '65', '68', '69', '72', '73', '74', '76', '77', '78', '80', '84', '88', '91', '94', '95', '96']


In [14]:
# ============================================================
# Demand & Calls Trend — line chart on Part Lookup, keyed off $B$4 (P/N) and $D$4 (Branch)
# ============================================================

def branch_aware_sumifs(value_col_letter):
    """
    Sum `value_col_letter` in the pmovdcE sheet for the selected P/N ($B$4).
    If $D$4 = "National", sums across every branch; otherwise restricts to the
    selected Brc as well.
    """
    pn_range = f"'{MOVEMENT_SHEET_NAME}'!$B$2:$B${mov_last_row}"
    brc_range = f"'{MOVEMENT_SHEET_NAME}'!$A$2:$A${mov_last_row}"
    val_range = f"'{MOVEMENT_SHEET_NAME}'!${value_col_letter}$2:${value_col_letter}${mov_last_row}"
    national_sum = f"SUMIFS({val_range}, {pn_range}, $B$4)"
    branch_sum = f"SUMIFS({val_range}, {pn_range}, $B$4, {brc_range}, $D$4)"
    return f'=IF($D$4="National", {national_sum}, {branch_sum})'


trend_section_row = chart_anchor_row + 21   # clears the units trend chart above it

ws.merge_cells(f"A{trend_section_row}:Q{trend_section_row}")
set_cell(ws, f"A{trend_section_row}", "Call & Demand", section_font, section_fill)

header_row_t = trend_section_row + 1
set_cell(ws, f"A{header_row_t}", "Months Ago", label_font, lightgrey_fill, Alignment(horizontal="center"), brd=border)
# Left-to-right = oldest -> most recent, so the line chart reads chronologically
periods = list(range(N_DEMAND_MONTHS, 0, -1))   # 16, 15, ..., 1
for i, p in enumerate(periods):
    col = get_column_letter(2 + i)
    set_cell(ws, f"{col}{header_row_t}", p, label_font, month_fill, Alignment(horizontal="center"), brd=border)

# pmovdcE column layout is fixed by construction: col 1=Brc, 2=P/N, 3..18=D-1..D-16, 19..30=C-1..C-12
demand_row_t = header_row_t + 1
set_cell(ws, f"A{demand_row_t}", "Demand", label_font, lightgrey_fill, Alignment(horizontal="left", indent=1), brd=border)
for i, p in enumerate(periods):
    col = get_column_letter(2 + i)
    demand_col_letter = get_column_letter(2 + p)   # matches mov_headers layout
    formula = branch_aware_sumifs(demand_col_letter)
    set_cell(ws, f"{col}{demand_row_t}", formula, value_font, None, Alignment(horizontal="center"),
              numfmt="#,##0", brd=border)

calls_row_t = demand_row_t + 1
set_cell(ws, f"A{calls_row_t}", "Calls", label_font, lightgrey_fill, Alignment(horizontal="left", indent=1), brd=border)
for i, p in enumerate(periods):
    col = get_column_letter(2 + i)
    if p <= N_CALL_MONTHS:   # C-i only goes back N_CALL_MONTHS months
        call_col_letter = get_column_letter(2 + N_DEMAND_MONTHS + p)
        formula = branch_aware_sumifs(call_col_letter)
        set_cell(ws, f"{col}{calls_row_t}", formula, value_font, None, Alignment(horizontal="center"),
                  numfmt="#,##0", brd=border)
    else:
        set_cell(ws, f"{col}{calls_row_t}", "", value_font, None, Alignment(horizontal="center"), brd=border)

# chart — dual axis so Demand and Calls are visually comparable despite different scales
trend_cats = Reference(ws, min_col=2, max_col=1 + N_DEMAND_MONTHS, min_row=header_row_t, max_row=header_row_t)

demand_chart = LineChart()
demand_chart.title = "Demand & Calls Trend"
demand_chart.style = 2
demand_chart.y_axis.title = "Demand (D-i)"
demand_chart.x_axis.title = "Months Ago"
demand_chart.width = 24
demand_chart.height = 10
demand_chart.y_axis.crosses = "min"

demand_data = Reference(ws, min_col=1, max_col=1 + N_DEMAND_MONTHS, min_row=demand_row_t, max_row=demand_row_t)
demand_chart.add_data(demand_data, titles_from_data=True, from_rows=True)
demand_chart.set_categories(trend_cats)

calls_chart = LineChart()
calls_chart.y_axis.axId = 200
calls_chart.y_axis.title = "Calls (C-i)"
calls_chart.y_axis.crosses = "max"

calls_data = Reference(ws, min_col=1, max_col=1 + N_DEMAND_MONTHS, min_row=calls_row_t, max_row=calls_row_t)
calls_chart.add_data(calls_data, titles_from_data=True, from_rows=True)
calls_chart.set_categories(trend_cats)

demand_chart += calls_chart   # merges calls_chart onto demand_chart as a secondary axis

ws.add_chart(demand_chart, f"A{trend_section_row + 4}")

for col in "LMNOPQ":
    ws.column_dimensions[col].width = 13


In [15]:
wb.save(OUTPUT_FILE)
print(f"Saved: {OUTPUT_FILE}")
print(f"Sheet order: {wb.sheetnames}")


Saved: FD Processed Non Steron CNH Dashboard 19Aug26.xlsx
Sheet order: ['National Dashboard', 'Part Lookup', 'pmovdcE', 'FD Processed', 'DN Price Agc']


## Optional — verify formula values before opening in Excel

openpyxl never computes formula results, so any tool reading the file with
`data_only=True` (pandas, a quick preview, etc.) will see blank cells until the workbook has
been opened and saved once in real Excel, or recalculated headlessly with LibreOffice below.
**This step is optional** — opening `OUTPUT_FILE` directly in Excel recalculates everything
automatically.

In [16]:
# Requires LibreOffice ('soffice') installed locally — skip this cell if you don't have it.
# import subprocess
# result = subprocess.run(["soffice", "--headless", "--convert-to", "xlsx",
#                           "--outdir", ".", OUTPUT_FILE], capture_output=True, text=True)
# print(result.stdout, result.stderr)
